# Ingeniería Analítica — Click&Buy Latam
Notebook documentado del Hito 1. El script reproducible se encuentra en `src/`.

## Objetivo
Data wrangling: deduplicar por clave primaria, tratar nulos con criterio documentado, normalizar strings, parsear fechas y detectar outliers con IQR/Z-score.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

df = pd.read_csv(
    ROOT / "data/raw/unidad6-grupo1-ecommerce-retail.csv",
    dtype={"ID_Transaccion": "string"}
)

df.shape

(2040, 12)

## Auditoría inicial

In [2]:
df.info()
print(df.isna().sum())
print('IDs repetidos:',df['ID_Transaccion'].duplicated().sum())
print('Duplicados exactos:',df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2040 entries, 0 to 2039
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID_Transaccion   2040 non-null   string 
 1   Fecha            2040 non-null   object 
 2   ID_Cliente       2040 non-null   object 
 3   Categoria        1960 non-null   object 
 4   Producto         2040 non-null   object 
 5   Precio_Unitario  2040 non-null   object 
 6   Cantidad         2040 non-null   int64  
 7   Descuento_Pct    1940 non-null   float64
 8   Monto_Total_USD  2040 non-null   float64
 9   Metodo_Pago      2040 non-null   object 
 10  Canal_Venta      2040 non-null   object 
 11  Estado_Pedido    2040 non-null   object 
dtypes: float64(2), int64(1), object(8), string(1)
memory usage: 191.4+ KB
ID_Transaccion       0
Fecha                0
ID_Cliente           0
Categoria           80
Producto             0
Precio_Unitario      0
Cantidad             0
Descuento_Pct

## Criterios de tratamiento
- Deduplicación por `ID_Transaccion`; grupos repetidos se verifican por conflicto.
- Categoría se infiere desde producto usando relaciones observadas.
- Descuento se imputa 0 solo si el monto confirma precio × cantidad.
- Cantidades negativas se conservan y marcan.
- Outliers se detectan por IQR (1.5×IQR) y Z-score (|z|>3), sin borrarlos automáticamente.

## Ejecutar el pipeline
Desde la raíz del proyecto: `python src/03_limpieza_ingenieria_analitica.py`.

In [3]:
import subprocess,sys
subprocess.run([sys.executable,str(ROOT/'src/03_limpieza_ingenieria_analitica.py')],check=True)

Filas: 2040 -> 2000 | duplicados por ID eliminados: 40
Categorías completadas: 80 | descuentos imputados 0: 100
Fechas inválidas: 0 | montos inconsistentes: 0
Nulos finales: 0 | CSV: /tmp/clickbuy_notebook_verify/Analisis-dev/data/processed/dataset_limpio.csv


CompletedProcess(args=['/opt/pyvenv/bin/python', '/tmp/clickbuy_notebook_verify/Analisis-dev/src/03_limpieza_ingenieria_analitica.py'], returncode=0)

## Validación final

In [4]:
clean=pd.read_csv(ROOT/'data/processed/dataset_limpio.csv')
print(clean.shape)
print('IDs únicos:',clean['ID_Transaccion'].nunique())
print('Nulos:',clean.isna().sum().sum())
print('Montos inconsistentes:',(~clean['Monto_Consistente']).sum())
clean.head()

(2000, 23)
IDs únicos: 2000
Nulos: 0
Montos inconsistentes: 0


,ID_Transaccion,Fecha,ID_Cliente,Categoria,Producto,Precio_Unitario,Cantidad,Descuento_Pct,Monto_Total_USD,Metodo_Pago,Canal_Venta,Estado_Pedido,Categoria_Imputada,Descuento_Imputado,Diferencia_Monto_Calculado_USD,Monto_Consistente,Cantidad_Negativa,Outlier_IQR_Precio,Outlier_ZScore_Precio,Outlier_IQR_Cantidad,Outlier_ZScore_Cantidad,Outlier_IQR_Monto,Outlier_ZScore_Monto
0,TX-G1-10000,2025-11-24,CLI-395,Electrónica,"Smart TV 55""",423.41,1,0.00,423.41,Transferencia,Marketplace,Entregado,True,True,0.0,True,False,False,False,False,False,False,False
1,TX-G1-10001,2025-02-27,CLI-568,Electrónica,Auriculares Bluetooth,616.21,1,0.20,492.97,Tarjeta Crédito,Marketplace,Entregado,False,False,0.0,True,False,False,False,False,False,False,False
2,TX-G1-10002,2025-01-13,CLI-347,Hogar,Set de Sartenes,229.55,1,0.05,218.07,Transferencia,App Mobile,Entregado,False,False,-0.0,True,False,False,False,False,False,False,False
3,TX-G1-10003,2025-05-21,CLI-187,Belleza,Secador de Pelo,33.94,1,0.20,27.15,Tarjeta Crédito,Sitio Web,En Tránsito,False,False,-0.0,True,False,False,False,False,False,False,False
4,TX-G1-10004,2025-05-06,CLI-144,Indumentaria,Pantalón Jean,58.69,1,0.00,58.69,MercadoPago,App Mobile,Entregado,False,False,0.0,True,False,False,False,False,False,False,False


## Entregables
Dataset limpio, script reproducible, auditoría resumen, matriz variable por variable, ficha técnica y guía para las siguientes duplas. El EDA y dashboard corresponden a los roles posteriores.